In [6]:
import re, html, pandas as pd, spacy

df = pd.read_csv("../data/processed/clean.csv")
df = df.dropna(subset=["clean"])
print(df.shape)                       # (79788, 7)

splitter = spacy.blank("en")
splitter.add_pipe("sentencizer")

ASPECTS = {
    "pacing":     ["pace", "pacing", "paced", "slow", "drag", "dragged", "drags", "rushed"],
    "characters": ["character", "characters", "protagonist", "heroine", "narrator"],
    "plot":       ["plot", "storyline", "story", "twist", "twists"],
    "ending":     ["ending", "endings", "finale", "epilogue"],
    "writing":    ["writing", "prose", "style", "narration", "writer","well written", "well-written", "poorly written","badly written", "beautifully written"],
    "emotion":    ["emotional", "emotions", "heartbreaking", "touching", "tears", "cried"],
}
ASPECT_RE = {a: re.compile(r"\b(" + "|".join(kws) + r")\b") for a, kws in ASPECTS.items()}

(79788, 7)


In [7]:
sub = df.sample(10_000, random_state=42).reset_index(drop=True)
titles = sub["title"].tolist()
labels = sub["label"].tolist()

rows = []
for i, doc in enumerate(splitter.pipe(sub["text"].map(html.unescape), batch_size=200)):
    for s in doc.sents:
        sent = s.text.strip()
        if len(sent.split()) < 4:
            continue
        low = sent.lower()
        for aspect, rx in ASPECT_RE.items():
            if rx.search(low):
                rows.append((i, titles[i], labels[i], aspect, sent))

sents = pd.DataFrame(rows, columns=["review_id", "title", "review_label", "aspect", "sentence"])
print(len(sents))
print(sents["aspect"].value_counts())

11595
aspect
plot          4477
characters    3202
writing       2540
emotion        522
pacing         431
ending         423
Name: count, dtype: int64


In [3]:
for a in ASPECTS:
    print(a.upper())
    for t in sents[sents["aspect"] == a]["sentence"].sample(3, random_state=1):
        print("  -", t[:200])

PACING
  - By the end of this book, it struck me that it's very much like a lot of historical and military fiction: decent at getting across the salient facts, decent at putting the reader in the middle of the a
  - Fast paced action revolving around a real weapon system.
  - He has the skill to smoothly draw the reader into an entertaining story, and then to slowly put the reader off-guard until you find yourself wrapped up in the middle of a very dark narrative.
CHARACTERS
  - I can't remember the last time I read a book that began with a two-page list of characters!
  - So what were my issues?In Northanger Abbey, the heroine is also fairly naive and kind of creates the problems in her life.
  - I have begun to make myself another character in the book!
PLOT
  - THIS STORY WAS VERY CAPTIVATING AND EASY TO READ,UNDERSTAND AND FOLLOW.I'VE ALWAYS LOVED THIS STORY AND NOW I DO EVEN MORE THAT BEFORE.HOWERVER, THE ENDING WAS QUITE DIFFERENT THEN I REMEMBER.
  - Harold Bell Wright weaves an

In [5]:
from transformers import pipeline
import time

clf = pipeline("text-classification",
               model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
               device=-1)

t0 = time.time()
clf(sents["sentence"].head(200).tolist(), batch_size=32, truncation=True, max_length=128)
secs = time.time() - t0
print(round(secs, 1), "seconds for 200 sentences")
print("estimated minutes for all:", round(secs / 200 * len(sents) / 60, 1))

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1030.28it/s]


10.7 seconds for 200 sentences
estimated minutes for all: 11.5


In [8]:
out = clf(sents["sentence"].tolist(), batch_size=32, truncation=True, max_length=128)
sents["positive"] = [o["label"] == "POSITIVE" for o in out]
sents.to_csv("../data/processed/aspect_sentences.csv", index=False)
print(sents["positive"].mean())

0.6930573523070289


In [9]:
summary = sents.groupby("aspect").agg(sentences=("positive", "size"),
                                      pct_positive=("positive", "mean"))
summary["pct_positive"] = (summary["pct_positive"] * 100).round(1)
print(summary)

# how often each aspect appears in negative vs positive reviews
n_by_label = sub["label"].value_counts()
mention = (sents.drop_duplicates(["review_id", "aspect"])
                .groupby(["aspect", "review_label"]).size().unstack(fill_value=0))
mention_pct = (mention / n_by_label * 100).round(1)
mention_pct.columns = ["% of negative reviews", "% of positive reviews"]
print(mention_pct)

            sentences  pct_positive
aspect                             
characters       3202          66.6
emotion           522          75.5
ending            423          54.4
pacing            431          54.5
plot             4477          70.9
writing          2540          73.7
            % of negative reviews  % of positive reviews
aspect                                                  
characters                   24.4                   19.8
emotion                       1.9                    4.9
ending                        4.5                    3.7
pacing                        4.9                    3.8
plot                         27.6                   27.2
writing                      23.3                   18.7
